### Load the data and prepare the plotting tables


In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

DATA_PATH = r"..."
CACHE_PATH = r"..."

USECOLS = [
    "City",
    "Restaurant Name",
    "Recommendations",
    "Rating (Restaurant Level)",
    "Address",
    "Rating % (Review Level)",
    "Rating Stars (Review Level)",
    "Date",
    "Postleitzahl",
    "Word Count",
    "Sentiment",
    "ID",
    "ZIP Code",
    "Region",
    "Review Topic",
    "Review Language",
    "Gender Indicator"
]

if os.path.exists(CACHE_PATH):
    df = pd.read_pickle(CACHE_PATH)
else:
    df = pd.read_csv(
        DATA_PATH,
        usecols=lambda col: col.strip() in USECOLS,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
        dtype={"ID": str},
        on_bad_lines="skip"
    )
    df.to_pickle(CACHE_PATH)

df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_cols = [
    "Rating Stars (Review Level)",
    "Rating % (Review Level)",
    "Rating (Restaurant Level)",
    "Recommendations",
    "Word Count"
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

# I keep one clean table for reviews and another one for topic-level plots.
review_df = df.drop_duplicates(subset="ID").copy()
topic_df = df.copy()

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set2


### Sentiment distribution across reviews


In [ ]:
plot_df = (
    review_df["Sentiment"]
    .dropna()
    .value_counts()
    .reset_index()
)

plot_df.columns = ["Sentiment", "Number of Reviews"]

total_reviews = plot_df["Number of Reviews"].sum()
plot_df["Share"] = plot_df["Number of Reviews"] / total_reviews

plot_df["Label"] = (
    plot_df["Number of Reviews"].astype(int).astype(str)
    + " ("
    + plot_df["Share"].apply(lambda x: f"{x:.1%}")
    + ")"
)

sentiment_colors = {
    "positive": "#2ca25f",
    "negative": "#de2d26",
    "neutral": "#f1c40f"
}

fig = px.bar(
    plot_df,
    x="Sentiment",
    y="Number of Reviews",
    color="Sentiment",
    color_discrete_map=sentiment_colors,
    text="Label",
    title="Distribution of Review Sentiment",
    labels={
        "Sentiment": "Sentiment Category",
        "Number of Reviews": "Number of Reviews"
    },
)

fig.update_traces(
    textposition="outside",
    marker_line_color="white",
    marker_line_width=1.5
)

fig.update_layout(
    title_x=0.5,
    showlegend=False,
    font=dict(size=14),
    height=520,
    yaxis_title="Number of Reviews",
    xaxis_title="Sentiment",
    plot_bgcolor="white",
    paper_bgcolor="white",
    bargap=0.25,
    uniformtext_minsize=10,
    uniformtext_mode="show"
)

fig.show()


### Most frequent review topics


In [ ]:
plot_df = (
    topic_df["Review Topic"]
    .dropna()
    .value_counts()
    .reset_index()
)

plot_df.columns = ["Review Topic", "Topic Mentions"]

fig = px.bar(
    plot_df,
    x="Topic Mentions",
    y="Review Topic",
    orientation="h",
    color="Review Topic",
    text="Topic Mentions",
    title="Frequency of Review Topics",
    labels={"Topic Mentions": "Number of Topic Mentions", "Review Topic": "Review Topic"},
)

fig.update_traces(textposition="outside")
fig.update_layout(
    title_x=0.5,
    showlegend=False,
    font=dict(size=14),
    height=560,
    yaxis=dict(categoryorder="total ascending"),
)

fig.show()


### Sentiment composition by review topic


In [ ]:
plot_df = topic_df.dropna(subset=["Review Topic", "Sentiment"]).copy()

plot_df["Sentiment"] = plot_df["Sentiment"].astype(str).str.strip().str.lower()

sentiment_order = ["negativ", "neutral", "positiv"]

sentiment_colors = {
    "positive": "#2ca25f",
    "negative": "#de2d26",
    "neutral": "#f1c40f"
}
plot_df = (
    plot_df
    .groupby(["Review Topic", "Sentiment"])
    .size()
    .reset_index(name="Count")
)

plot_df["Share"] = (
    plot_df["Count"] /
    plot_df.groupby("Review Topic")["Count"].transform("sum")
)

plot_df["Share_Label"] = plot_df["Share"].apply(lambda x: f"{x:.1%}")

fig = px.bar(
    plot_df,
    x="Review Topic",
    y="Share",
    color="Sentiment",
    text="Share_Label",
    color_discrete_map=sentiment_colors,
    category_orders={"Sentiment": sentiment_order},
    title="Sentiment Composition Across Review Topics",
    labels={
        "Review Topic": "Review Topic",
        "Share": "Share of Topic Mentions",
        "Sentiment": "Sentiment",
    },
)

fig.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(
        color="white",
        size=13
    ),
    marker_line_color="white",
    marker_line_width=1.2
)

fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=580,
    yaxis_tickformat=".0%",
    legend_title_text="Sentiment",
    plot_bgcolor="white",
    paper_bgcolor="white",
    bargap=0.25,
    uniformtext_minsize=10,
    uniformtext_mode="show"
)

fig.show()


### Monthly review volume


In [ ]:
plot_df = review_df.dropna(subset=["Date"]).copy()
plot_df["Month"] = plot_df["Date"].dt.to_period("M").dt.to_timestamp()

plot_df = (
    plot_df
    .groupby("Month", as_index=False)
    .agg(Number_of_Reviews=("ID", "nunique"))
)

fig = px.line(
    plot_df,
    x="Month",
    y="Number_of_Reviews",
    markers=True,
    title="Monthly Review Volume Over Time",
    labels={
        "Month": "Month",
        "Number_of_Reviews": "Number of Reviews",
    },
)

fig.update_traces(line=dict(width=3), marker=dict(size=7))
fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=560,
    hovermode="x unified",
)

fig.show()


### Monthly sentiment composition


In [ ]:
plot_df = review_df.dropna(subset=["Date", "Sentiment"]).copy()

plot_df["Sentiment"] = plot_df["Sentiment"].astype(str).str.strip().str.lower()
plot_df["Month"] = plot_df["Date"].dt.to_period("M").dt.to_timestamp()

sentiment_order = ["negativ", "neutral", "positiv"]

sentiment_colors = {
    "positive": "#2ca25f",
    "negative": "#de2d26",
    "neutral": "#f1c40f"
}

plot_df = (
    plot_df
    .groupby(["Month", "Sentiment"])
    .agg(Number_of_Reviews=("ID", "nunique"))
    .reset_index()
)

plot_df["Share"] = (
    plot_df["Number_of_Reviews"] /
    plot_df.groupby("Month")["Number_of_Reviews"].transform("sum")
)

fig = px.area(
    plot_df,
    x="Month",
    y="Share",
    color="Sentiment",
    color_discrete_map=sentiment_colors,
    category_orders={"Sentiment": sentiment_order},
    title="Monthly Sentiment Composition Over Time",
    labels={
        "Month": "Month",
        "Share": "Share of Reviews",
        "Sentiment": "Sentiment",
    },
)

fig.update_traces(
    line=dict(width=1.5),
    opacity=0.85
)

fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=580,
    yaxis_tickformat=".0%",
    hovermode="x unified",
    legend_title_text="Sentiment",
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.show()


### Monthly topic composition


In [ ]:
plot_df = topic_df.dropna(subset=["Date", "Review Topic"]).copy()
plot_df["Month"] = plot_df["Date"].dt.to_period("M").dt.to_timestamp()

# Keep each topic only once per review.
plot_df = plot_df.drop_duplicates(subset=["Month", "ID", "Review Topic"])

plot_df = (
    plot_df
    .groupby(["Month", "Review Topic"], as_index=False)
    .agg(Topic_Mentions=("ID", "nunique"))
)

plot_df["Total_Monthly_Topic_Mentions"] = (
    plot_df
    .groupby("Month")["Topic_Mentions"]
    .transform("sum")
)

plot_df["Topic_Share"] = (
    plot_df["Topic_Mentions"] / plot_df["Total_Monthly_Topic_Mentions"]
)

topic_colors = {
    "Food": "#1b9e77",
    "Service": "#d95f02",
    "Price": "#7570b3",
    "Ambiance": "#e7298a",
    "General/Other": "#66a61e"
}

topic_order = ["Food", "Service", "Price", "Ambiance", "General/Other"]

fig = px.line(
    plot_df,
    x="Month",
    y="Topic_Share",
    color="Review Topic",
    markers=True,
    color_discrete_map=topic_colors,
    category_orders={"Review Topic": topic_order},
    title="Monthly Topic Composition Over Time",
    labels={
        "Month": "Month",
        "Topic_Share": "Share of Monthly Topic Mentions",
        "Review Topic": "Review Topic",
    },
)

fig.update_traces(
    line=dict(width=2.8),
    marker=dict(size=5)
)

fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=600,
    width=1150,
    yaxis_tickformat=".0%",
    hovermode="x unified",
    legend_title_text="Review Topic",
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.show()


### Top 20 cities by review volume


In [ ]:
plot_df = (
    review_df
    .dropna(subset=["City"])
    .groupby("City", as_index=False)
    .agg(Number_of_Reviews=("ID", "nunique"))
    .sort_values("Number_of_Reviews", ascending=False)
    .head(20)
    .sort_values("Number_of_Reviews", ascending=True)
)

fig = px.bar(
    plot_df,
    x="Number_of_Reviews",
    y="City",
    orientation="h",
    color="Number_of_Reviews",
    text="Number_of_Reviews",
    color_continuous_scale="Blues",
    title="Top 20 Cities by Number of Reviews",
    labels={
        "Number_of_Reviews": "Number of Reviews",
        "City": "City",
    },
)

fig.update_traces(textposition="outside")
fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=700,
    coloraxis_showscale=False,
)

fig.show()


### Regional sentiment composition


In [ ]:
plot_df = review_df.dropna(subset=["Region", "Sentiment"]).copy()

region_counts = (
    plot_df
    .groupby("Region")
    .agg(Total_Reviews=("ID", "nunique"))
    .reset_index()
)

plot_df = plot_df.merge(region_counts, on="Region", how="left")
plot_df = plot_df[plot_df["Total_Reviews"] >= 20]

plot_df = (
    plot_df
    .groupby(["Region", "Sentiment"])
    .agg(Number_of_Reviews=("ID", "nunique"))
    .reset_index()
)

plot_df["Share"] = (
    plot_df["Number_of_Reviews"] /
    plot_df.groupby("Region")["Number_of_Reviews"].transform("sum")
)

heatmap_df = plot_df.pivot(
    index="Region",
    columns="Sentiment",
    values="Share"
).fillna(0)

fig = px.imshow(
    heatmap_df,
    text_auto=".1%",
    aspect="auto",
    color_continuous_scale="RdYlGn",
    title="Regional Sentiment Composition Heatmap",
    labels=dict(
        x="Sentiment",
        y="Region",
        color="Share of Reviews"
    ),
)

fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=700,
)

fig.show()


### Review length by sentiment


In [ ]:
plot_df = review_df.dropna(subset=["Sentiment", "Word Count"]).copy()
plot_df = plot_df[plot_df["Word Count"] > 0]

plot_df["Sentiment"] = plot_df["Sentiment"].astype(str).str.strip().str.lower()

sentiment_order = ["negativ", "neutral", "positiv"]

sentiment_colors = {
    "positive": "#2ca25f",
    "negative": "#de2d26",
    "neutral": "#f1c40f"
}

fig = px.violin(
    plot_df,
    x="Sentiment",
    y="Word Count",
    color="Sentiment",
    color_discrete_map=sentiment_colors,
    category_orders={"Sentiment": sentiment_order},
    box=True,
    points="outliers",
    title="Distribution of Review Length by Sentiment",
    labels={
        "Sentiment": "Sentiment",
        "Word Count": "Review Length in Words",
    },
)

fig.update_traces(
    marker_line_color="white",
    marker_line_width=0.8,
    opacity=0.85
)

fig.update_layout(
    title_x=0.5,
    showlegend=False,
    font=dict(size=14),
    height=600,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis_title="Sentiment",
    yaxis_title="Review Length in Words"
)

fig.show()


### Sentiment by gender indicator


In [ ]:
plot_df = review_df.dropna(subset=["Gender Indicator", "Sentiment"]).copy()

# I group the different manual gender labels first.
plot_df["Gender Indicator"] = plot_df["Gender Indicator"].astype(str).str.strip()

gender_map = {
    "Male": "Male",
    "Female": "Female",
    "Unknown": "Unknown"
}

plot_df["Gender_clean"] = plot_df["Gender Indicator"].str.lower().str.strip()
plot_df["Gender_Grouped"] = plot_df["Gender_clean"].map(gender_map).fillna("Unclear")

plot_df["Sentiment"] = plot_df["Sentiment"].astype(str).str.strip().str.lower()

plot_df = (
    plot_df
    .groupby(["Gender_Grouped", "Sentiment"])
    .agg(Number_of_Reviews=("ID", "nunique"))
    .reset_index()
)

plot_df["Total_Reviews"] = (
    plot_df
    .groupby("Gender_Grouped")["Number_of_Reviews"]
    .transform("sum")
)

plot_df["Share"] = plot_df["Number_of_Reviews"] / plot_df["Total_Reviews"]

gender_order = ["Female", "Male", "Unclear"]
sentiment_order = ["negativ", "neutral", "positiv"]

heatmap_df = (
    plot_df
    .pivot(index="Gender_Grouped", columns="Sentiment", values="Share")
    .fillna(0)
)

existing_genders = [g for g in gender_order if g in heatmap_df.index]
other_genders = [g for g in heatmap_df.index if g not in existing_genders]
heatmap_df = heatmap_df.reindex(existing_genders + other_genders)

existing_sentiments = [s for s in sentiment_order if s in heatmap_df.columns]
other_sentiments = [s for s in heatmap_df.columns if s not in existing_sentiments]
heatmap_df = heatmap_df[existing_sentiments + other_sentiments]

fig = px.imshow(
    heatmap_df,
    text_auto=".1%",
    aspect="auto",
    color_continuous_scale="RdYlGn",
    title="Sentiment Composition by Gender Indicator",
    labels=dict(
        x="Sentiment",
        y="Gender Indicator",
        color="Share of Reviews"
    ),
)

fig.update_traces(
    textfont=dict(
        color="white",
        size=15
    )
)

fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=500,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis_title="Sentiment",
    yaxis_title="Gender Indicator"
)

fig.show()


### Topic distribution in the top 10 cities


In [ ]:
top_cities = (
    review_df
    .dropna(subset=["City"])
    .groupby("City", as_index=False)
    .agg(Number_of_Reviews=("ID", "nunique"))
    .sort_values("Number_of_Reviews", ascending=False)
    .head(10)
)

city_order = top_cities["City"].tolist()

plot_df = topic_df.dropna(subset=["City", "Review Topic", "ID"]).copy()

plot_df = plot_df[plot_df["City"].isin(city_order)]

# Keep each topic only once per review.
plot_df = plot_df.drop_duplicates(subset=["City", "ID", "Review Topic"])

plot_df = (
    plot_df
    .groupby(["City", "Review Topic"], as_index=False)
    .agg(Topic_Mentions=("ID", "nunique"))
)

plot_df["Total_Topic_Mentions"] = (
    plot_df
    .groupby("City")["Topic_Mentions"]
    .transform("sum")
)

plot_df["Topic_Share"] = plot_df["Topic_Mentions"] / plot_df["Total_Topic_Mentions"]

plot_df = plot_df.merge(top_cities, on="City", how="left")

topic_colors = {
    "Food": "#1b9e77",
    "Service": "#d95f02",
    "Price": "#7570b3",
    "Ambiance": "#e7298a",
    "General/Other": "#66a61e"
}

topic_order = ["Food", "Service", "Price", "Ambiance", "General/Other"]

fig = px.bar(
    plot_df,
    x="City",
    y="Topic_Share",
    color="Review Topic",
    barmode="group",
    text=plot_df["Topic_Share"].apply(lambda x: f"{x:.1%}"),
    color_discrete_map=topic_colors,
    category_orders={
        "City": city_order,
        "Review Topic": topic_order
    },
    title="Topic Distribution in the Top 10 Cities by Review Volume",
    labels={
        "City": "City",
        "Topic_Share": "Share of Topic Mentions",
        "Review Topic": "Review Topic"
    },
    hover_data={
        "Topic_Mentions": True,
        "Total_Topic_Mentions": True,
        "Number_of_Reviews": True,
        "Topic_Share": ":.1%"
    }
)

fig.update_traces(
    textposition="outside",
    textangle=0,
    marker_line_color="white",
    marker_line_width=1.2
)

fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=700,
    width=1000,
    yaxis_tickformat=".0%",
    yaxis_title="Share of Topic Mentions",
    xaxis_title="Top 10 Cities",
    xaxis_tickangle=-30,
    legend_title_text="Review Topic",
    plot_bgcolor="white",
    paper_bgcolor="white",
    bargap=0.18,
    bargroupgap=0.06,
    uniformtext_minsize=9,
    uniformtext_mode="hide"
)

fig.show()


### Topic distribution in the top 10 regions


In [ ]:
top_regions = (
    review_df
    .dropna(subset=["Region"])
    .groupby("Region", as_index=False)
    .agg(Number_of_Reviews=("ID", "nunique"))
    .sort_values("Number_of_Reviews", ascending=False)
    .head(10)
)

region_order = top_regions["Region"].tolist()

plot_df = topic_df.dropna(subset=["Region", "Review Topic", "ID"]).copy()

plot_df = plot_df[plot_df["Region"].isin(region_order)]

# Keep each topic only once per review.
plot_df = plot_df.drop_duplicates(subset=["Region", "ID", "Review Topic"])

plot_df = (
    plot_df
    .groupby(["Region", "Review Topic"], as_index=False)
    .agg(Topic_Mentions=("ID", "nunique"))
)

plot_df["Total_Topic_Mentions"] = (
    plot_df
    .groupby("Region")["Topic_Mentions"]
    .transform("sum")
)

plot_df["Topic_Share"] = plot_df["Topic_Mentions"] / plot_df["Total_Topic_Mentions"]

plot_df = plot_df.merge(top_regions, on="Region", how="left")

topic_colors = {
    "Food": "#1b9e77",
    "Service": "#d95f02",
    "Price": "#7570b3",
    "Ambiance": "#e7298a",
    "General/Other": "#66a61e"
}

topic_order = ["Food", "Service", "Price", "Ambiance", "General/Other"]

fig = px.bar(
    plot_df,
    x="Region",
    y="Topic_Share",
    color="Review Topic",
    barmode="group",
    text=plot_df["Topic_Share"].apply(lambda x: f"{x:.1%}"),
    color_discrete_map=topic_colors,
    category_orders={
        "Region": region_order,
        "Review Topic": topic_order
    },
    title="Topic Distribution in the Top 10 Regions by Review Volume",
    labels={
        "Region": "Region / Bundesland",
        "Topic_Share": "Share of Topic Mentions",
        "Review Topic": "Review Topic"
    },
    hover_data={
        "Topic_Mentions": True,
        "Total_Topic_Mentions": True,
        "Number_of_Reviews": True,
        "Topic_Share": ":.1%"
    }
)

fig.update_traces(
    textposition="outside",
    textangle=0,
    marker_line_color="white",
    marker_line_width=1.2
)

fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=700,
    width=1000,
    yaxis_tickformat=".0%",
    yaxis_title="Share of Topic Mentions",
    xaxis_title="Top 10 Regions / Bundesländer",
    xaxis_tickangle=-35,
    legend_title_text="Review Topic",
    plot_bgcolor="white",
    paper_bgcolor="white",
    bargap=0.18,
    bargroupgap=0.06,
    uniformtext_minsize=9,
    uniformtext_mode="hide"
)

fig.show()


### Topic distribution by gender indicator


In [ ]:
plot_df = topic_df.dropna(subset=["Gender Indicator", "Review Topic", "ID"]).copy()

plot_df["Gender Indicator"] = plot_df["Gender Indicator"].astype(str).str.strip()
plot_df["Gender_clean"] = plot_df["Gender Indicator"].str.lower().str.strip()

gender_map = {
    "Male": "Male",
    "Female": "Female",
    "Unknown": "Unknown"
}

plot_df["Gender_Grouped"] = plot_df["Gender_clean"].map(gender_map).fillna("Unclear")

# Keep each topic only once per review.
plot_df = plot_df.drop_duplicates(subset=["Gender_Grouped", "ID", "Review Topic"])

plot_df = (
    plot_df
    .groupby(["Gender_Grouped", "Review Topic"], as_index=False)
    .agg(Topic_Mentions=("ID", "nunique"))
)

plot_df["Total_Topic_Mentions"] = (
    plot_df
    .groupby("Gender_Grouped")["Topic_Mentions"]
    .transform("sum")
)

plot_df["Topic_Share"] = plot_df["Topic_Mentions"] / plot_df["Total_Topic_Mentions"]

gender_review_counts = (
    review_df
    .dropna(subset=["Gender Indicator"])
    .copy()
)

gender_review_counts["Gender Indicator"] = gender_review_counts["Gender Indicator"].astype(str).str.strip()
gender_review_counts["Gender_clean"] = gender_review_counts["Gender Indicator"].str.lower().str.strip()
gender_review_counts["Gender_Grouped"] = gender_review_counts["Gender_clean"].map(gender_map).fillna("Unclear")

gender_review_counts = (
    gender_review_counts
    .groupby("Gender_Grouped", as_index=False)
    .agg(Number_of_Reviews=("ID", "nunique"))
)

plot_df = plot_df.merge(gender_review_counts, on="Gender_Grouped", how="left")

topic_colors = {
    "Food": "#1b9e77",
    "Service": "#d95f02",
    "Price": "#7570b3",
    "Ambiance": "#e7298a",
    "General/Other": "#66a61e"
}

topic_order = ["Food", "Service", "Price", "Ambiance", "General/Other"]
gender_order = ["Female", "Male", "Unclear"]

fig = px.bar(
    plot_df,
    x="Gender_Grouped",
    y="Topic_Share",
    color="Review Topic",
    barmode="group",
    text=plot_df["Topic_Share"].apply(lambda x: f"{x:.1%}"),
    color_discrete_map=topic_colors,
    category_orders={
        "Gender_Grouped": gender_order,
        "Review Topic": topic_order
    },
    title="Topic Distribution by Gender Indicator",
    labels={
        "Gender_Grouped": "Gender Indicator",
        "Topic_Share": "Share of Topic Mentions",
        "Review Topic": "Review Topic"
    },
    hover_data={
        "Topic_Mentions": True,
        "Total_Topic_Mentions": True,
        "Number_of_Reviews": True,
        "Topic_Share": ":.1%"
    }
)

fig.update_traces(
    textposition="outside",
    textangle=0,
    marker_line_color="white",
    marker_line_width=1.2
)

fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    height=650,
    width=1050,
    yaxis_tickformat=".0%",
    yaxis_title="Share of Topic Mentions",
    xaxis_title="Gender Indicator",
    legend_title_text="Review Topic",
    plot_bgcolor="white",
    paper_bgcolor="white",
    bargap=0.20,
    bargroupgap=0.07,
    uniformtext_minsize=9,
    uniformtext_mode="hide"
)

fig.show()
